# H-006 · Book size N

N ∈ {20, 35, 50} on the frozen cadence. Freeze `N_STAR`. Slot weight `1/N`.

Do not open the sealed holdout in this notebook.

Next: run H-007.


## 0. Imports & Config


In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s2_coint.report import (
    arm_selection_table,
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    plot_fold_boxplots,
    require_star,
    save_star_stack,
    write_tearsheet_pdf,
)
from backtest.s2_coint.research import (
    config_from_stack,
    is_end_for_stack,
    load_s1_weekly,
    load_s2_cuts,
    load_universe_panels,
    lookbacks_for_bar,
    n_trials_local,
    n_trials_stack,
    overlay_ols_hedge,
    overlay_kalman_hedge,
    register_hypothesis_arms,
    s2_artifacts_dir,
    s2_data_dir,
    tearsheet_path,
)
from backtest.s2_coint.runner import run_s2_backtest
from backtest.s2_coint.walkforward import embargo_bars_for_config, make_s2_folds

VERSION = "v2"
STAR_PATH = os.path.join(s2_artifacts_dir(version=VERSION, root=ROOT), "s2_star_stack.json")
LEDGER_PATH = os.path.join(os.path.dirname(STAR_PATH), "s2_variant_ledger.json")
cuts = load_s2_cuts(root=ROOT, version=VERSION)
stack = load_star_stack(STAR_PATH)
print("yahoo_drop_rate", cuts.get("yahoo_drop_rate"))
print("stack keys", sorted(stack.keys()))

from backtest.s2_coint.research import candidates_path, rebalance_path
from strategies.s2_coint.book import select_book

HYP_ID = "H-006"
N_ARMS = (20, 35, 50)
require_star("BOOK_STAR", stack.get("BOOK_STAR"))

require_star("POOL_STAR", stack.get("POOL_STAR"))
require_star("FDR_STAR", stack.get("FDR_STAR"))
PAIRS_STAR = list(stack.get("PAIRS_STAR") or [])
if not PAIRS_STAR:
    raise ValueError("PAIRS_STAR empty — freeze H-001/H-002 first")
BAR = str(stack.get("BAR_STAR") or "1d")
train, full = load_universe_panels(None, BAR, PAIRS_STAR, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(BAR)
train = overlay_ols_hedge(train, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                          hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
full = overlay_ols_hedge(full, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                         hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
is_end = is_end_for_stack(stack, full if BAR == "1h" else train)
if BAR == "1d":
    is_panel, oos_panel = train.copy(), full.loc[pd.to_datetime(full["date"]) > is_end].copy()
else:
    from backtest.s2_coint.research import split_is_oos
    is_panel, oos_panel = split_is_oos(full, is_end=is_end)
s1_weekly = load_s1_weekly(ROOT)
print("bar", BAR, "is_end", is_end, "IS", len(is_panel), "OOS", len(oos_panel), "pairs", len(PAIRS_STAR))



## 1. Build books by N and score


In [ ]:
cand = pd.read_csv(candidates_path(BAR, version=VERSION, root=ROOT))
POOL_STAR = str(stack["POOL_STAR"])
if POOL_STAR == "industry":
    base = cand.loc[cand["same_industry"].astype(bool)].copy()
else:
    base = cand.loc[cand["same_sector"].astype(bool)].copy()
if str(stack.get("FDR_STAR")) == "bh_fdr":
    from backtest.s2_coint.research import apply_bh_fdr
    base = apply_bh_fdr(base.loc[base["eligible"].astype(bool)].copy(), alpha=0.05)
    base = base.loc[base["fdr_pass"]].copy()
else:
    base = base.loc[base["eligible"].astype(bool) & (base["pvalue"] < 0.05)].copy()
leaf = "industry_y" if POOL_STAR == "industry" else "sector_y"
pool_map = {str(r.pair_id): str(getattr(r, leaf)) for r in base.itertuples(index=False)}

books = {}
for n in N_ARMS:
    chosen = select_book(base, pool_of_pair=pool_map, per_pool_cap=max(n, 10), global_cap=n, pvalue_threshold=1.0)
    books[str(n)] = [str(p) for p in chosen["pair_id"]]
    print("N", n, "->", len(books[str(n)]))

union = sorted(set().union(*[set(v) for v in books.values()]))
train_u, full_u = load_universe_panels(None, BAR, union, root=ROOT, version=VERSION)
lb0 = lookbacks_for_bar(BAR)
is_panel = overlay_ols_hedge(train_u, ols_window=lb0["ols_window"], z_window=lb0["z_window"],
                             hl_window=lb0["hl_window"], adf_window=lb0["adf_window"])
s1_weekly = load_s1_weekly(ROOT)
dates = pd.DatetimeIndex(pd.to_datetime(is_panel["date"])).sort_values().unique()
folds = make_s2_folds(dates, n_folds=3, embargo_bars=embargo_bars_for_config(bar=BAR))
display(fold_table(folds))
register_hypothesis_arms(HYP_ID, [str(n) for n in N_ARMS], overwrite=True, path=LEDGER_PATH)
fold_parts, full_parts = [], []
for n in N_ARMS:
    arm = str(n)
    p = is_panel.loc[is_panel["pair_id"].astype(str).isin(set(books[arm]))].copy()
    cfg = {arm: config_from_stack(stack)}
    fold_parts.append(fold_val_metrics(p, folds, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
    full_parts.append(full_is_metrics(p, cfg, hyp_id=HYP_ID, s1_weekly=s1_weekly))
fold_df = pd.concat(fold_parts, ignore_index=True)
full_is_df = pd.concat(full_parts, ignore_index=True)
plot_fold_boxplots(fold_df, title="H-006 N fold-val")
plt.show()
print("median-Sharpe hint (commentary only):", median_sharpe_hint(fold_df))
display(arm_selection_table(fold_df, full_is_df))


## 2. Type `N_STAR`


In [ ]:
N_STAR = ""  # type after the table: "20" | "35" | "50"
require_star("N_STAR", N_STAR)
stack["N_STAR"] = int(N_STAR)
stack["PAIRS_STAR"] = books[str(N_STAR)]
from backtest.s2_coint.research import pairs_path
pd.DataFrame({"pair_id": books[str(N_STAR)], "rank": range(1, len(books[str(N_STAR)]) + 1)}).to_csv(
    pairs_path(BAR, version=VERSION, root=ROOT), index=False
)
save_star_stack(STAR_PATH, stack)
print("wrote", STAR_PATH, "N_STAR", N_STAR)


## 3. Notes for next hyp

Next: run H-007.
